# 16. Reward Model and RL Intro

## Problem

为什么 SFT 之后还常常不够？为什么“语言上通顺”和“人真正更喜欢”不是同一个目标？奖励模型、偏好数据、RLHF、DPO 又是怎样连起来的？

## Dependencies

建议先理解 SFT、assistant-style 数据、log probability 和基本生成流程。


## Why SFT is not the end

SFT 的强项是给模型一个清晰的行为骨架，但它仍然有局限。

面对同一个问题，可能会存在多个都“看起来合理”的回答：

- 一个回答更直接
- 一个回答更完整
- 一个回答更安全
- 一个回答更有条理

这些回答在语言上都可能通顺，但人类偏好并不一样。

这时训练目标就不再只是“像示范那样回答”，而是要回答：

- 在多个候选里，哪一个更值得保留？
- 哪一种风格更有帮助？
- 哪一种回答更符合产品或安全偏好？

于是后面就会引入 preference data 和 reward model。它们的任务不是替代 SFT，而是在 SFT 之后继续把模型往更符合偏好的方向推。


## Pairwise preference objective

奖励模型常见的一种训练方式，是拿同一个 prompt 的两个回答做比较：

- $y^{+}$：被人类认为更好的回答
- $y^{-}$：被人类认为更差的回答

然后训练一个评分函数 $r_\phi(x, y)$，让它给 chosen 回答更高分。常见公式是：

$$
L_{RM} = -\log \sigma\big(r_\phi(x, y^{+}) - r_\phi(x, y^{-})\big)
$$

这里：

- $x$ 是输入 prompt。
- $r_\phi(x, y)$ 是 reward model 对回答 $y$ 的分数。
- 如果 chosen 分数明显高于 rejected，损失就会变小。

这个目标的直觉非常重要：reward model 不是在判断“绝对真理”，而是在拟合**相对偏好**。


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# 一个最小 pairwise preference 示例。
# 假设对于同一个 prompt，人类更喜欢 chosen，而不是 rejected。
chosen_score = 2.4
rejected_score = 1.1

# Bradley-Terry / logistic 风格的 pairwise loss。
# 如果 chosen_score 比 rejected_score 高很多，loss 会变小。
pairwise_loss = -np.log(1.0 / (1.0 + np.exp(-(chosen_score - rejected_score))))

print('chosen score =', chosen_score)
print('rejected score =', rejected_score)
print('pairwise preference loss =', pairwise_loss)


## Reward model is a scorer, not a truth oracle

很多人会误解 reward model，以为它是在判断回答“对不对”。这不准确。

更准确的说法是：reward model 在学习一种偏好映射，它试图把“什么样的回答更受偏好”压缩成一个分数。

这个分数可能综合了很多因素：

- 直接性
- 完整性
- 安全性
- 格式感
- 帮助性

所以 reward model 更像一个**偏好打分器**，不是一个真理裁判。

这也是它的风险所在：如果偏好数据本身有偏差，或者 reward model 只学到了表面模式，它就可能把模型推向“更像高分模板”，而不是真的更有帮助。


In [ ]:
# 一个玩具 reward model 打分例子。
# 三个特征只用于演示：directness, completeness, safety。

answers = np.array([
    [0.9, 0.7, 0.8],
    [0.4, 0.8, 0.6],
    [0.7, 0.5, 0.9],
])

reward_w = np.array([0.5, 0.3, 0.6])
rewards = answers @ reward_w

print('feature matrix =')
print(answers)
print('reward scores =', rewards)
print('best answer index under this toy reward model =', rewards.argmax())


## From reward model to RLHF and DPO

把 reward model 放回完整链路里看，会更清楚：

1. **Pretraining**：先建立广义语言能力。
2. **SFT**：先把模型拉到更像助手的回答分布。
3. **Reward model**：把 preference signal 压成可优化的分数。
4. **Policy optimization**：再让主模型朝更高偏好方向更新。

如果走 RLHF 路线，一个常见直觉是：

- 让模型生成候选回答
- reward model 给这些回答打分
- 再用 PPO 一类方法更新 policy，同时控制不要偏离 reference model 太远

一个很常见的简化写法是：

$$
R_{total} = R_{rm} - \beta \cdot KL\big(\pi_\theta \;\|\; \pi_{ref}\big)
$$

这里的意思是：

- $R_{rm}$ 希望 policy 朝更高偏好分数走
- $KL$ 惩罚 policy 跑得太远，避免训练不稳定或回答风格崩掉

如果走 DPO 路线，则通常不显式训练 value function，也不做完整 RL rollout，而是直接在 preference pairs 上更新 policy。它常见的一种写法是：

$$
L_{DPO} = -\log \sigma\Big(\beta \big[(\log \pi_\theta(y^{+}|x) - \log \pi_\theta(y^{-}|x)) - (\log \pi_{ref}(y^{+}|x) - \log \pi_{ref}(y^{-}|x))\big]\Big)
$$

你可以把它理解成：**不仅要让 policy 更偏向 chosen，而且这种偏向要相对 reference model 更明显。**


In [ ]:
# 一个最小 DPO 风格例子。
# 我们比较 policy 和 reference model 对 chosen / rejected 的相对偏好差。

policy_logprob_chosen = -1.2
policy_logprob_rejected = -2.0
ref_logprob_chosen = -1.5
ref_logprob_rejected = -1.8
beta = 0.1

policy_gap = policy_logprob_chosen - policy_logprob_rejected
ref_gap = ref_logprob_chosen - ref_logprob_rejected
dpo_margin = beta * (policy_gap - ref_gap)
dpo_loss = -np.log(1.0 / (1.0 + np.exp(dpo_margin)))

print('policy gap =', policy_gap)
print('reference gap =', ref_gap)
print('dpo margin =', dpo_margin)
print('dpo loss =', dpo_loss)

# 再给一个 PPO 风格的 toy reward 直觉。
rm_reward = 2.3
kl_penalty = 0.4
beta_kl = 0.2
total_reward = rm_reward - beta_kl * kl_penalty

print('ppo-style total reward =', total_reward)


## Common mistakes

- 把 reward model 理解成“知识真假判断器”。它更准确地说是在拟合偏好信号。
- 以为 RLHF 或 DPO 会让模型突然学会新知识。它们更多是在已有能力之上调整行为方向。
- 忽略偏好数据质量。偏好优化是否有效，很大程度取决于 preference signal 本身。
- 只看分数提升，不看 reward hacking 风险。模型可能学会讨好 reward，而不是真的更有帮助。

## Summary

这一阶段的核心，不再是“把下一个 token 猜准”或者“模仿示范回答”，而是进一步回答：**在多个可行回答里，哪一个更值得保留？**

reward model 提供偏好分数，RLHF / PPO 提供基于回报的优化路线，DPO 提供更直接的 preference optimization 路线。它们共同服务的目标，是把已经会回答的模型继续推向更符合目标偏好的行为分布。
